In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, ttest_ind

np.random.seed(42)

# -----------------------------
# Synthetic Dataset
# -----------------------------
data = {
    "time_slot": ["6-9","6-9","9-13","13-17","17-20","20-23","23-2"],
    "avg_speed_mbps": [180,170,150,120,90,70,160]
}

real_data = pd.DataFrame(data)
real_speeds = real_data["avg_speed_mbps"].values

# -----------------------------
# Monte Carlo Simulation
# -----------------------------
def simulate(n=5000):
    results = []

    for _ in range(n):
        slot = np.random.choice(["offpeak","moderate","peak"])

        if slot == "offpeak":
            users = np.random.poisson(20)
            capacity = np.random.normal(200, 15)
            alpha = 0.01
        elif slot == "moderate":
            users = np.random.poisson(50)
            capacity = np.random.normal(180, 20)
            alpha = 0.02
        else:
            users = np.random.poisson(80)
            capacity = np.random.normal(150, 25)
            alpha = 0.03

        users = max(users, 1)
        efficiency = np.random.uniform(0.8, 1.2)
        noise = np.random.normal(0, 5)

        speed = (capacity * efficiency) / (1 + alpha * users) + noise
        results.append(speed)

    return np.array(results)

simulated = simulate()

# -----------------------------
# Statistics
# -----------------------------
print("Real Mean:", np.mean(real_speeds))
print("Sim Mean:", np.mean(simulated))

# KS Test
ks_stat, p_val = ks_2samp(real_speeds, simulated)
print("KS p-value:", p_val)

# T-test (peak approximation)
t_stat, p_val2 = ttest_ind(real_speeds, simulated[:len(real_speeds)])
print("T-test p-value:", p_val2)

# -----------------------------
# Visualization
# -----------------------------
plt.hist(real_speeds, alpha=0.6, label="Baseline")
plt.hist(simulated, alpha=0.6, label="Monte Carlo")
plt.legend()
plt.title("ISP Bandwidth Distribution Comparison")
plt.show()